# Exp 12 - Replay and Spoofing Attack Detection using IDS in Python

> Scope note: this notebook is a lab-scale deterministic simulation. It is intended for understanding timing, communication, and security concepts. It is not a certification model for a production autonomous vehicle.

## Objective

Build a simple intrusion-detection model for replay and spoofing attacks in V2X messages.

Replay detection checks whether a nonce has already been seen. Spoofing detection checks whether the authentication tag is invalid. A physical-plausibility rule catches values that are unlikely even if message structure looks correct.

## Textbook Notes and Case Studies

### 1. Textbook Background

Replay and spoofing attacks are major threats in connected vehicle systems. A replay attack resends a previously valid message. A spoofing attack sends a message with a false identity, false location, false speed, or false event claim.

An Intrusion Detection System observes messages and raises alerts when behavior violates expected rules. IDS logic can be signature-based, anomaly-based, specification-based, or hybrid. A simple lab IDS may check timestamps, duplicate message IDs, impossible movement, invalid speed, or mismatched identity.

### 2. Architecture Notes

```
Incoming V2X Message Stream
        |
        v
Feature Extraction: sender, timestamp, position, speed, nonce
        |
        v
IDS Rules / Anomaly Detection
        |
        v
Alert Type: Replay / Spoofing / Normal
```

The IDS should not be judged only by the number of alerts. False positives and false negatives are both important. A noisy IDS may be ignored, while a weak IDS may miss real attacks.

### 3. Important Metrics

True positive:

```
attack correctly detected
```

False positive:

```
normal message incorrectly flagged
```

False negative:

```
attack not detected
```

Detection rate:

```
detection_rate = true_positives / actual_attacks
```

False positive rate:

```
false_positive_rate = false_positives / normal_messages
```

Replay check example:

```
flag if timestamp <= last_seen_timestamp for same sender
```

### 4. Classroom Case Studies

Case Study A - Replay of Emergency Brake:
An attacker records a valid emergency-brake message and rebroadcasts it later. Timestamp, nonce, or sequence-number validation should reject the old message.

Case Study B - Spoofed Vehicle Position:
A malicious sender claims to be at an intersection when it is not. Plausibility checks can compare claimed movement with maximum feasible speed.

Case Study C - Sybil-Like Identity Abuse:
One physical attacker pretends to be multiple vehicles. A simple lab IDS may not fully detect this, showing the limitation of rule-only detection.

### 5. Analysis Checklist

Report the rule used, messages flagged, attack type, and detection metrics. Clearly identify limitations: simple rules can be bypassed, and real systems need authentication, certificate checks, sensor consistency, and behavior analysis.

### 6. Source Notes

- NIST guidance on system risk and threat analysis: https://csrc.nist.gov/publications/detail/sp/800-30/rev-1/final
- ETSI intelligent transport security standards page: https://www.etsi.org/technologies/automotive-intelligent-transport


## Architecture

```text
Incoming V2X Packet
  |-- sender id
  |-- nonce
  |-- speed
  |-- timestamp
  |-- tag
          |
          v
IDS Rule Engine
  |-- replay rule: nonce already seen?
  |-- spoof rule: HMAC invalid?
  |-- plausibility rule: speed too high?
          |
          v
Alert Labels
  |-- NORMAL
  |-- REPLAY
  |-- SPOOFING
  |-- PHYSICAL-PLAUSIBILITY
```

## Formulas and Required Theory

Replay rule:

\[
\text{replay} =
\begin{cases}
1, & nonce \in SeenNonces\\
0, & otherwise
\end{cases}
\]

Spoofing rule:

\[
\text{spoofing} = \neg compare\_digest(tag_{received}, tag_{computed})
\]

Risk score used in the post-lab:

\[
\text{score} = \sum \text{rule weights for triggered alerts}
\]

## In-Lab Method

1. Create sample V2X packets.
2. Compute valid HMAC tags for normal packets.
3. Inject one replayed nonce.
4. Inject one spoofed tag.
5. Run IDS rules and print alert labels.

In [1]:
import hashlib
import hmac

print("EXP 12 - IN-LAB IDS DETECTION")
key = b"ids-key"
seen_nonces = set()
packets = [
    {"id": "V1", "nonce": "n1", "speed": 40, "ts": 100},
    {"id": "V1", "nonce": "n2", "speed": 42, "ts": 101},
    {"id": "V1", "nonce": "n1", "speed": 40, "ts": 100},
    {"id": "V9", "nonce": "x1", "speed": 180, "ts": 102},
]
for pkt in packets:
    text = f"{pkt['id']}|{pkt['nonce']}|{pkt['speed']}|{pkt['ts']}".encode()
    pkt["tag"] = hmac.new(key, text, hashlib.sha256).hexdigest()
packets[-1]["tag"] = "spoofed"

for pkt in packets:
    text = f"{pkt['id']}|{pkt['nonce']}|{pkt['speed']}|{pkt['ts']}".encode()
    valid = hmac.compare_digest(pkt["tag"], hmac.new(key, text, hashlib.sha256).hexdigest())
    replay = pkt["nonce"] in seen_nonces
    seen_nonces.add(pkt["nonce"])
    alerts = []
    if replay:
        alerts.append("REPLAY")
    if not valid:
        alerts.append("SPOOFING")
    if pkt["speed"] > 130:
        alerts.append("PHYSICAL-PLAUSIBILITY")
    print(pkt["id"], pkt["nonce"], alerts or ["NORMAL"])

EXP 12 - IN-LAB IDS DETECTION
V1 n1 ['NORMAL']
V1 n2 ['NORMAL']
V1 n1 ['REPLAY']
V9 x1 ['SPOOFING', 'PHYSICAL-PLAUSIBILITY']


## Post-Lab Method

The post-lab cell assigns weights to IDS labels and converts each event into low, medium, or high severity.

In [2]:
print("EXP 12 - POST-LAB IDS RULE SCORING")
events = [
    ["NORMAL"],
    ["REPLAY"],
    ["SPOOFING"],
    ["SPOOFING", "PHYSICAL-PLAUSIBILITY"],
    ["REPLAY", "SPOOFING"],
]
weights = {"REPLAY": 4, "SPOOFING": 6, "PHYSICAL-PLAUSIBILITY": 3, "NORMAL": 0}
for i, event in enumerate(events, 1):
    score = sum(weights[e] for e in event)
    severity = "high" if score >= 7 else "medium" if score >= 4 else "low"
    print(f"event={i} labels={event} score={score} severity={severity}")

EXP 12 - POST-LAB IDS RULE SCORING
event=1 labels=['NORMAL'] score=0 severity=low
event=2 labels=['REPLAY'] score=4 severity=medium
event=3 labels=['SPOOFING'] score=6 severity=medium
event=4 labels=['SPOOFING', 'PHYSICAL-PLAUSIBILITY'] score=9 severity=high
event=5 labels=['REPLAY', 'SPOOFING'] score=10 severity=high


## What to Write in the Lab Record

- Include the alert output.
- Explain the difference between replay and spoofing.
- State why nonces/timestamps are needed.
- Explain how weighted IDS scoring helps prioritise response.

## References

- Python `hmac` documentation: https://docs.python.org/3/library/hmac.html
- Python `hashlib` documentation: https://docs.python.org/3/library/hashlib.html